In [1]:
# imports and settings

import os
import gpxpy
import time
import torch
import librosa
import pickle
import warnings
import pandas as pd
import networkx as nx

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch, get_window
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import utils as ut
from reference_detector.ship_noise_pkl_model import run_model as run_reference_model

# do not show warnings
warnings.filterwarnings("ignore")

# autoreload for development
%load_ext autoreload
%autoreload 2

print("Imports complete.")

Imports complete.
Settings: height=800, width=1400, font_size=16
Imports complete.


In [2]:
folder_path = '../data/croatia/2407_1_600m'
file_list = os.listdir(folder_path)
file_list = [f for f in file_list if f.endswith('.wav')]
file_list.sort()

data = []
for file in file_list[6:-15]:
    # print(f"Loading file: {file}")
    _data, fs = librosa.load(os.path.join(folder_path, file))
    data.append(_data)
data = np.concatenate(data)
# data = data[80*fs:]
print(f"data final shape: {data.shape} of duration: {len(data)/fs} seconds")

data final shape: (51597000,) of duration: 2340.0 seconds


In [3]:
gpx_file = os.path.join(folder_path, '600m_route.gpx')
gpx_data = gpxpy.parse(open(gpx_file))

# get the first track
track = gpx_data.tracks[0]

# get the first segment
segment = track.segments[0]

rx_location = gpxpy.gpx.GPXWaypoint(44.330008, 14.697141)

distances = []
for point in segment.points:
    distance = rx_location.distance_2d(point)
    distances.append(distance)

distances = np.array(distances)
t_distances = np.linspace(0, len(data)/fs, len(distances))

In [4]:
nperseg = 1000
overlap = 0.5
window = 'hanning'
dc = 20
crop_freq = 8000
norm_size = 5
s2g_quantization_levels = 3
s2g_mode = 'wasserstein'
welch_threshold = 0.034
s2g_threshold = 0.3
reference_model_threshold = 0.5

best_detector = ut.BestDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
talmon_detector = ut.TalmonDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, norm_size=norm_size)
s2g_detector = ut.S2GDetector(fs, nperseg=nperseg, overlap=overlap, window=window, dc=dc, crop_freq=crop_freq, mode=s2g_mode, quantization_levels=s2g_quantization_levels)

In [7]:
# sanity - check spectral power + K only for f0 = 631, 859
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

f0_1_bd_score = []
f0_2_bd_score = []

f0_1_s2g_scores = []
f0_2_s2g_scores = []

f0_1_td_scores = []
f0_2_td_scores = []

f0_1 = 631  # Hz
f0_2 = 859  # Hz
step_size = 10 * fs

for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    pxx, F_pxx = best_detector.get_feature_vector(_data)
    f0_1_pxx = pxx[np.where(F_pxx >= f0_1)[0][0]]
    f0_2_pxx = pxx[np.where(F_pxx >= f0_2)[0][0]]
    f0_1_bd_score.append(f0_1_pxx)
    f0_2_bd_score.append(f0_2_pxx)

    K, F_K = s2g_detector.get_feature_vector(_data)
    f0_1_K_pxx = K[np.where(F_K >= f0_1)[0][0]]
    f0_2_K_pxx = K[np.where(F_K >= f0_2)[0][0]]
    f0_1_s2g_scores.append(f0_1_K_pxx)
    f0_2_s2g_scores.append(f0_2_K_pxx)

    td_scores, td_F = talmon_detector.get_feature_vector(_data)
    f0_1_td_pxx = td_scores[np.where(td_F >= f0_1)[0][0]]
    f0_2_td_pxx = td_scores[np.where(td_F >= f0_2)[0][0]]
    f0_1_td_scores.append(f0_1_td_pxx)
    f0_2_td_scores.append(f0_2_td_pxx)

detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))
print("completed all detectors")

completed all detectors


In [10]:
fig = make_subplots(rows=4, 
                    cols=1, 
                    row_heights=[0.7, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "distance", "Talmon", "S2G"])

fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance', marker_color='black', showlegend=False), row=2, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_bd_score, name=f'Welch f={f0_1} Hz', marker_color='blue'), row=3, col=1)
# fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_bd_score, name=f'Welch f={f0_2} Hz', marker_color='cyan'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_td_scores, name=f'Talmon f={f0_1} Hz', marker_color='blue', showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_td_scores, name=f'Talmon f={f0_2} Hz', marker_color='green', showlegend=False), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_1_s2g_scores, name=f'Score f={f0_1} Hz', marker_color='blue'), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=f0_2_s2g_scores, name=f'Score f={f0_2} Hz', marker_color='green'), row=4, col=1)


fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="Dist. (m)", row=2, col=1)
fig.update_yaxes(title_text="Score", row=3, col=1)
fig.update_yaxes(title_text="Score", row=4, col=1)
# fig.update_yaxes(title_text="Score", row=5, col=1)

fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, scores for f0 = {f0_1}, {f0_2} Hz")
fig.show(renderer="browser")
fig.write_html("../results/nb84_analysis_dpv_600m_f0(631).html")

___

In [ ]:
_data = data[700*fs:760*fs]
F, T, Sxx, phasogram = ut.calc_spectrogram(_data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

pxx, F_pxx = best_detector.get_feature_vector(_data)
K, F_K = s2g_detector.get_feature_vector(_data)
td_scores, td_F = talmon_detector.get_feature_vector(_data)

fig = make_subplots(rows=1, cols=4, shared_yaxes=True, column_widths=[0.7, 0.1, 0.1, 0.1])
fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=pxx, y=F_pxx, name='Best Detector', marker_color='blue'), row=1, col=2)
fig.add_trace(go.Scatter(x=K, y=F_K, name='S2G Detector', marker_color='green'), row=1, col=3)
fig.add_trace(go.Scatter(x=td_scores, y=td_F, name='Talmon Detector', marker_color='red'), row=1, col=4)
fig.update_xaxes(title_text="Time (s)", row=1, col=1)
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.show(renderer="browser")


In [11]:
F, T, Sxx, phasogram = ut.calc_spectrogram(data, fs=fs, nperseg=nperseg, percent_overlap=overlap, window=window, remove_dc=dc, crop_freq=crop_freq)

max_welch_score = []
max_s2g_score = []
max_td_score = []

step_size = 10 * fs
for si in np.arange(start=0, stop=(len(data)-60*fs), step=step_size):
    _data = data[si:si+60*fs]
    _time = (si + len(_data) / 2) // fs  # time corresponding to the center of the window

    pxx, F_pxx = best_detector.get_feature_vector(_data)
    max_welch_score.append(np.max(pxx))

    K, F_K = s2g_detector.get_feature_vector(_data)
    max_s2g_score.append(np.max(K))

    td_scores, td_F = talmon_detector.get_feature_vector(_data)
    max_td_score.append(np.max(td_scores))
    
detectors_taxis = np.linspace(T[0], T[-1], len(f0_1_s2g_scores))

In [12]:
fig = make_subplots(rows=5, 
                    cols=1, 
                    row_heights=[0.6, 0.1, 0.1, 0.1, 0.1], 
                    vertical_spacing=0.01, 
                    shared_xaxes=True,
                    row_titles=["Spectrogram", "distance", "Best Detector", "S2G"])

fig.add_trace(go.Heatmap(z=Sxx, x=T, y=F, colorscale='viridis', showlegend=False, showscale=False), row=1, col=1)
fig.add_trace(go.Scatter(x=t_distances, y=distances, name='Distance (m)', marker_color='black'), row=2, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=max_welch_score, name='Welch Detector max score', marker_color='blue'), row=3, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=max_s2g_score, name='S2G Detector max score', marker_color='purple'), row=4, col=1)
fig.add_trace(go.Scatter(x=detectors_taxis, y=max_td_score, name='Talmon Detector max score', marker_color='green'), row=5, col=1)

fig.update_xaxes(title_text="Time (s)", row=4, col=1)
fig.update_yaxes(title_text="Frequency (Hz)", row=1, col=1)
fig.update_yaxes(title_text="(m)", row=2, col=1)

fig.update_layout(height=800, width=1200, title_text=f"Scooter experiment 600m, max scores")
fig.show(renderer="browser")
fig.write_html("../results/nb84_analysis_dpv_600m_max_scores.html")